In [64]:
import pandas as pd
import numpy as np
import os

# load master data - takes 3 seconds now
master_df = pd.read_parquet(r"D:\CricMetric-AI\data\processed\master_odi.parquet")

print(f"Shape: {master_df.shape}")
print(f"Columns: {master_df.columns.tolist()}")

Shape: (1349408, 30)
Columns: ['innings', 'over', 'batting_team', 'batter', 'non_striker', 'bowler', 'runs_off_bat', 'extras', 'wides', 'noballs', 'byes', 'legbyes', 'wicket_type', 'player_dismissed', 'match_id', 'match_date', 'team1', 'team2', 'venue', 'event', 'toss_winner', 'city', 'season', 'over_num', 'ball_num', 'is_wicket', 'total_runs_ball', 'cumulative_runs', 'cumulative_wickets', 'year']


In [65]:
cols_to_drop = [
    'extra1', 'extra2', 'extra3',
    'running_total', 'penalty',
    'other_wicket_type', 'other_player_dismissed'
]
cols_to_drop = [col for col in cols_to_drop if col in master_df.columns]
master_df = master_df.drop(columns=cols_to_drop)

print(f"Remaining columns: {master_df.shape[1]}")
print(master_df.columns.tolist())

Remaining columns: 30
['innings', 'over', 'batting_team', 'batter', 'non_striker', 'bowler', 'runs_off_bat', 'extras', 'wides', 'noballs', 'byes', 'legbyes', 'wicket_type', 'player_dismissed', 'match_id', 'match_date', 'team1', 'team2', 'venue', 'event', 'toss_winner', 'city', 'season', 'over_num', 'ball_num', 'is_wicket', 'total_runs_ball', 'cumulative_runs', 'cumulative_wickets', 'year']


In [66]:
# check null values in each column
null_counts = master_df.isnull().sum()
null_percent = (null_counts / len(master_df) * 100).round(2)

null_df = pd.DataFrame({
    'null_count': null_counts,
    'null_percent': null_percent
})

# only show columns that have nulls
null_df = null_df[null_df['null_count'] > 0]
print(null_df)

Empty DataFrame
Columns: [null_count, null_percent]
Index: []


In [67]:
# check for duplicate balls (same match, same over, same ball number)
duplicates = master_df.duplicated(
    subset=['match_id', 'over_num', 'ball_num', 'innings']
).sum()

print(f"Duplicate balls: {duplicates}")

Duplicate balls: 0


In [68]:

# every match → every innings → every over → every ball in order
master_df = master_df.sort_values(
    ['match_id', 'innings', 'over_num', 'ball_num']
).reset_index(drop=True)

print("Sorted successfully")
print(master_df[['match_id', 'innings', 'over_num', 'ball_num', 'batter', 'runs_off_bat']].head(10))

Sorted successfully
  match_id  innings  over_num  ball_num     batter  runs_off_bat
0  1000887        1         0         1  DA Warner             0
1  1000887        1         0         2  DA Warner             0
2  1000887        1         0         3  DA Warner             0
3  1000887        1         0         4  DA Warner             0
4  1000887        1         0         5  DA Warner             0
5  1000887        1         0         6  DA Warner             0
6  1000887        1         0         7  DA Warner             0
7  1000887        1         1         1    TM Head             0
8  1000887        1         1         2    TM Head             1
9  1000887        1         1         3  DA Warner             0


In [69]:
print(master_df['wicket_type'].value_counts().head(20))
print("\nUnique values sample:")
print(master_df['wicket_type'].unique()[:20])

wicket_type
""                       1312494
caught                     21118
bowled                      6915
lbw                         4102
run out                     2690
caught and bowled           1108
stumped                      882
retired hurt                  57
hit wicket                    35
obstructing the field          6
"caught                        1
Name: count, dtype: int64

Unique values sample:
<ArrowStringArray>
[                   '""',                'bowled',                'caught',
               'run out',                   'lbw',          'retired hurt',
               'stumped',     'caught and bowled',            'hit wicket',
 'obstructing the field',               '"caught']
Length: 11, dtype: str


In [70]:
# flag each ball as wicket or not
# recompute is_wicket correctly after sorting
master_df['is_wicket'] = master_df['wicket_type'].apply(
    lambda x: 1 if x not in ['', '""'] else 0
)

# verify - should show mostly 0s with occasional 1s
print("Wicket distribution:")
print(master_df['is_wicket'].value_counts())

# recompute cumulative wickets
master_df['cumulative_wickets'] = master_df.groupby(
    ['match_id', 'innings']
)['is_wicket'].cumsum()

# check first match again
print("\nFirst 15 balls:")
print(master_df[['match_id', 'innings', 'over_num', 'ball_num',
                  'batter', 'runs_off_bat', 'is_wicket',
                  'cumulative_runs', 'cumulative_wickets']].head(15))

# compute total runs per ball (batting runs + extras)
master_df['total_runs_ball'] = master_df['runs_off_bat'] + master_df['extras']

# cumulative runs and wickets within each innings of each match
master_df['cumulative_runs'] = master_df.groupby(
    ['match_id', 'innings']
)['total_runs_ball'].cumsum()


# verify
print(master_df[['match_id', 'innings', 'over_num', 'ball_num', 
                  'batter', 'runs_off_bat', 'is_wicket',
                  'cumulative_runs', 'cumulative_wickets']].head(15))

Wicket distribution:
is_wicket
0    1312494
1      36914
Name: count, dtype: int64

First 15 balls:
   match_id  innings  over_num  ball_num     batter  runs_off_bat  is_wicket  \
0   1000887        1         0         1  DA Warner             0          0   
1   1000887        1         0         2  DA Warner             0          0   
2   1000887        1         0         3  DA Warner             0          0   
3   1000887        1         0         4  DA Warner             0          0   
4   1000887        1         0         5  DA Warner             0          0   
5   1000887        1         0         6  DA Warner             0          0   
6   1000887        1         0         7  DA Warner             0          0   
7   1000887        1         1         1    TM Head             0          0   
8   1000887        1         1         2    TM Head             1          0   
9   1000887        1         1         3  DA Warner             0          0   
10  1000887        1

In [71]:
print("Wicket distribution:")
print(master_df['is_wicket'].value_counts())

Wicket distribution:
is_wicket
0    1312494
1      36914
Name: count, dtype: int64


In [72]:
master_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1349408 entries, 0 to 1349407
Data columns (total 30 columns):
 #   Column              Non-Null Count    Dtype         
---  ------              --------------    -----         
 0   innings             1349408 non-null  int64         
 1   over                1349408 non-null  str           
 2   batting_team        1349408 non-null  str           
 3   batter              1349408 non-null  str           
 4   non_striker         1349408 non-null  str           
 5   bowler              1349408 non-null  str           
 6   runs_off_bat        1349408 non-null  int64         
 7   extras              1349408 non-null  int64         
 8   wides               1349408 non-null  int64         
 9   noballs             1349408 non-null  int64         
 10  byes                1349408 non-null  int64         
 11  legbyes             1349408 non-null  int64         
 12  wicket_type         1349408 non-null  str           
 13  player_dismissed    134

In [73]:
# string to int conversions
int_cols = ['innings', 'over_num', 'ball_num', 'byes', 'legbyes']

for col in int_cols:
    master_df[col] = pd.to_numeric(master_df[col], errors='coerce').fillna(0).astype(int)

# match_date to datetime
master_df['match_date'] = pd.to_datetime(master_df['match_date'], errors='coerce')

# extract year as separate column
master_df['year'] = master_df['match_date'].dt.year

master_df.info()



<class 'pandas.DataFrame'>
RangeIndex: 1349408 entries, 0 to 1349407
Data columns (total 30 columns):
 #   Column              Non-Null Count    Dtype         
---  ------              --------------    -----         
 0   innings             1349408 non-null  int64         
 1   over                1349408 non-null  str           
 2   batting_team        1349408 non-null  str           
 3   batter              1349408 non-null  str           
 4   non_striker         1349408 non-null  str           
 5   bowler              1349408 non-null  str           
 6   runs_off_bat        1349408 non-null  int64         
 7   extras              1349408 non-null  int64         
 8   wides               1349408 non-null  int64         
 9   noballs             1349408 non-null  int64         
 10  byes                1349408 non-null  int64         
 11  legbyes             1349408 non-null  int64         
 12  wicket_type         1349408 non-null  str           
 13  player_dismissed    134

In [74]:
master_df.to_parquet(
    r"D:\CricMetric-AI\data\processed\master_odi.parquet",
    index=False
)
print("Saved cleaned dataframe successfully")
print(f"Shape: {master_df.shape}")

Saved cleaned dataframe successfully
Shape: (1349408, 30)


## Situation 1 — Early Collapse (PPI Component 1)

### What this measures
How ALL batters perform when team is in trouble early in the innings.
Specifically: every ball faced when 3+ wickets have already fallen
in first 20 overs of innings 1.

### Why innings 1 only
In innings 2 (chase), strike rate is context-dependent.
A batter scoring at 60 SR when required run rate is 5 is fine.
But in innings 1, any collapse situation requires scoring —
no free passes. SR is the right and only metric for innings 1.

### Filter conditions
| Condition | Value | Reason |
|---|---|---|
| cumulative_wickets | >= 3 | Team in collapse |
| over_num | < 20 | Early innings only |
| innings | == 1 | First innings pressure only |
| wides | == 0 | Legal deliveries only |

### Who gets counted
Both new batters AND set batters are counted.

Example:
- Team is 2 wickets down, Rohit batting, Kohli non-striker
- Kohli gets out → 3rd wicket falls → Dhoni walks in
- Ball after wicket: Dhoni facing → counted under Dhoni ✓
- Next ball: Rohit facing → counted under Rohit ✓
- Both are under pressure regardless of how long they batted

Non-striker is NEVER counted — they are not facing the ball.
Only the batter column (striker) gets credited.

### Design decision: counting all batters in pressure situation
Both new batters AND set batters are counted when
cumulative_wickets >= 3.
Reason: pressure is situational — any batter faces increased
mental and tactical pressure when team loses 3+ wickets,
regardless of how long they have been batting.
This aligns with CricViz's approach to pressure analysis.

### Metric used
Strike Rate under pressure = (runs / balls faced) × 100

In early collapse, scoring speed matters:
- A batter who scores at 80 SR stabilizes AND accelerates innings
- SR naturally punishes batters who get out cheaply
- SR naturally rewards batters who score despite pressure
- No separate dismissal count needed — already baked into SR

### Why SR and not average here
Average = runs per dismissal
SR = runs per ball

In collapse situations, BOTH survival and scoring speed matter.
But SR captures both better because:
- Getting out cheaply → low runs, same balls → low SR
- Surviving but not scoring → many balls, few runs → low SR
- Scoring quickly → many runs, fewer balls → high SR

### Minimum qualification
61+ balls faced in pressure situations.
Threshold = population average (60.9 balls per batter).
Statistically justified — batter must have faced at least
the average pressure exposure to qualify.
More defensible than arbitrary cutoff like 50.

### Key finding
248 batters qualified with 61+ pressure balls.
Legends like Dhoni (531 balls), Sangakkara (661 balls)
faced the most pressure — reflecting their middle order roles.
Kohli (164 balls) faced less — reflecting his top order position
where team rarely collapsed before he arrived.
Even with fewer balls, Kohli's SR in pressure reflects
how well he performed when collapse DID happen around him.

### Known limitation
Current implementation counts set batters AND new batters equally.
Ideally we would distinguish between:
- New batter walking in cold during collapse (harder)
- Set batter already there when collapse happens (slightly easier)
This distinction is a future improvement opportunity.
For now, situational pressure is treated equally for all batters
as per CricViz's industry standard approach.

In [75]:
# Situation 1: batting when 3+ wickets down in first 20 overs
pressure_s1 = master_df[
    (master_df['cumulative_wickets'] >= 3) &
    (master_df['over_num'] < 20) &
    (master_df['innings'] == 1) &
    (master_df['wides'] == 0)
].copy()

print(f"Pressure situation 1 balls: {len(pressure_s1)}")
print(f"Unique batters in pressure: {pressure_s1['batter'].nunique()}")

# compute stats for ALL batters first (before filtering)
s1_all = pressure_s1.groupby('batter').agg(
    s1_runs=('runs_off_bat', 'sum'),
    s1_balls=('runs_off_bat', 'count')
).reset_index()

s1_all['s1_sr'] = (
    s1_all['s1_runs'] / s1_all['s1_balls'] * 100
).round(2)

# statistical analysis BEFORE filtering
print("\nPressure balls distribution (all batters):")
print(s1_all['s1_balls'].describe())
print(f"Mean:             {s1_all['s1_balls'].mean():.2f}")
print(f"Median:           {s1_all['s1_balls'].median():.2f}")
print(f"25th percentile:  {s1_all['s1_balls'].quantile(0.25):.2f}")
print(f"75th percentile:  {s1_all['s1_balls'].quantile(0.75):.2f}")

# threshold = mean (population average)
min_s1_balls = int(s1_all['s1_balls'].mean())
print(f"Statistical threshold (mean): {min_s1_balls} balls")

s1_stats = s1_all[s1_all['s1_balls'] >= min_s1_balls].copy()

print(f"Batters qualifying ({min_s1_balls}+ pressure balls): {len(s1_stats)}")
print("\nTop 20 by strike rate under pressure:")
print(s1_stats.sort_values('s1_sr', ascending=False).head(20))

Pressure situation 1 balls: 53595
Unique batters in pressure: 880

Pressure balls distribution (all batters):
count    880.000000
mean      60.903409
std       96.369187
min        1.000000
25%        7.000000
50%       27.000000
75%       76.000000
max      738.000000
Name: s1_balls, dtype: float64
Mean:             60.90
Median:           27.00
25th percentile:  7.00
75th percentile:  76.00
Statistical threshold (mean): 60 balls
Batters qualifying (60+ pressure balls): 250

Top 20 by strike rate under pressure:
                    batter  s1_runs  s1_balls   s1_sr
162            CJ Anderson      159       108  147.22
364               JD Ryder      100        78  128.21
386             JM Davison       71        60  118.33
588            NLTC Perera      107        93  115.05
501            MDKJ Perera      105        98  107.14
201              DA Warner      118       115  102.61
121            BJ McMullen       77        78   98.72
165            CJ Ferguson       72        73   9

In [76]:
# check specific players
check_players = ['V Kohli', 'RG Sharma', 'MS Dhoni', 'AB de Villiers', 'KC Sangakkara']

for player in check_players:
    player_data = pressure_s1[pressure_s1['batter'] == player]
    print(f"{player}: {len(player_data)} pressure balls")

V Kohli: 164 pressure balls
RG Sharma: 202 pressure balls
MS Dhoni: 531 pressure balls
AB de Villiers: 323 pressure balls
KC Sangakkara: 661 pressure balls


In [77]:
# Situation 2: overs 40-50, chasing 270+, innings 2
# first find matches where team 2 needed 270+
innings2 = master_df[master_df['innings'] == 2].copy()

# get target for each match (max cumulative runs in innings 1 + 1)
innings1_totals = master_df[
    master_df['innings'] == 1
].groupby('match_id')['cumulative_runs'].max().reset_index()
innings1_totals.columns = ['match_id', 'innings1_total']
innings1_totals['target'] = innings1_totals['innings1_total'] + 1

# merge target into innings 2
innings2 = innings2.merge(innings1_totals, on='match_id', how='left')

# filter: overs 40+, target 270+, legal deliveries only
pressure_s2 = innings2[
    (innings2['over_num'] >= 40) &
    (innings2['target'] >= 270) &
    (innings2['wides'] == 0)
].copy()

print(f"Pressure situation 2 balls: {len(pressure_s2)}")
print(f"Unique batters: {pressure_s2['batter'].nunique()}")

# compute stats
s2_stats = pressure_s2.groupby('batter').agg(
    s2_runs=('runs_off_bat', 'sum'),
    s2_balls=('runs_off_bat', 'count')
).reset_index()

s2_stats['s2_sr'] = (s2_stats['s2_runs'] / s2_stats['s2_balls'] * 100).round(2)
s2_stats = s2_stats[s2_stats['s2_balls'] >= 50]

print(f"\nBatters with 30+ death chase balls: {len(s2_stats)}")
print("\nTop 10 by death chase strike rate:")
print(s2_stats.sort_values('s2_sr', ascending=False).head(10))

Pressure situation 2 balls: 33356
Unique batters: 931

Batters with 30+ death chase balls: 204

Top 10 by death chase strike rate:
             batter  s2_runs  s2_balls   s2_sr
511    MG Bracewell      151        81  186.42
496        MA Leask      110        61  180.33
814   Shahid Afridi      209       117  178.63
535      MP Stoinis      116        65  178.46
253    Fakhar Zaman      123        69  178.26
744       SB Styris      108        63  171.43
193   DAS Gunaratne      122        72  169.44
431    KP Pietersen      132        79  167.09
332  Iftikhar Ahmed       88        53  166.04
670      R Shepherd      108        66  163.64


In [78]:
print(innings2.shape)
print(innings2.columns.tolist())

(616034, 32)
['innings', 'over', 'batting_team', 'batter', 'non_striker', 'bowler', 'runs_off_bat', 'extras', 'wides', 'noballs', 'byes', 'legbyes', 'wicket_type', 'player_dismissed', 'match_id', 'match_date', 'team1', 'team2', 'venue', 'event', 'toss_winner', 'city', 'season', 'over_num', 'ball_num', 'is_wicket', 'total_runs_ball', 'cumulative_runs', 'cumulative_wickets', 'year', 'innings1_total', 'target']


In [79]:
# Count pressure innings for each batter
pressure_innings_per_batter = (
    pressure_s2.groupby("batter")["match_id"]
    .nunique()
    .reset_index(name="pressure_innings")
)

# Average pressure innings across all batters
avg_pressure_innings = pressure_innings_per_batter["pressure_innings"].median()

print("Average pressure innings per batter:", avg_pressure_innings)

# Optional: see all batters sorted
print(
    pressure_innings_per_batter.sort_values(
        "pressure_innings",
        ascending=False
    )
)

Average pressure innings per batter: 2.0
               batter  pressure_innings
541          MS Dhoni                32
675         RA Jadeja                29
549       Mahmudullah                22
551  Mashrafe Mortaza                19
886           V Kohli                19
..                ...               ...
531        MN van Wyk                 1
528          MN Nawaz                 1
6               A Nao                 1
905   WTS Porterfield                 1
903       WK McCallan                 1

[931 rows x 2 columns]


In [80]:
pressure_innings_per_batter["pressure_innings"].describe()

count    931.000000
mean       3.243824
std        3.393170
min        1.000000
25%        1.000000
50%        2.000000
75%        4.000000
max       32.000000
Name: pressure_innings, dtype: float64

In [81]:
threshold = pressure_innings_per_batter["pressure_innings"].quantile(0.75)

print(threshold)

4.0


In [82]:
# get match winner for each innings 2
innings2 = master_df[master_df['innings'] == 2].copy()

innings1_totals = master_df[
    master_df['innings'] == 1
].groupby('match_id')['cumulative_runs'].max().reset_index()
innings1_totals.columns = ['match_id', 'innings1_total']
innings1_totals['target'] = innings1_totals['innings1_total'] + 1

innings2 = innings2.merge(innings1_totals, on='match_id', how='left')

match_outcomes = innings2.groupby('match_id').agg(
    chasing_team=('batting_team', 'first'),
    final_runs=('cumulative_runs', 'max'),
    target=('target', 'first')
).reset_index()

# did chasing team win?
match_outcomes['chase_won'] = (
    match_outcomes['final_runs'] >= match_outcomes['target']
).astype(int)

print(f"Total chase matches: {len(match_outcomes)}")
print(f"Chases won: {match_outcomes['chase_won'].sum()}")
print(f"Chase win rate: {match_outcomes['chase_won'].mean()*100:.1f}%")

# updated filter - 250+ target
pressure_s2 = innings2[
    (innings2['over_num'] >= 40) &
    (innings2['target'] >= 250) &
    (innings2['wides'] == 0)
].copy()

print(f"\nPressure situation 2 balls: {len(pressure_s2)}")
print(f"Unique batters: {pressure_s2['batter'].nunique()}")

# merge win result into pressure_s2
pressure_s2 = pressure_s2.merge(
    match_outcomes[['match_id', 'chase_won']],
    on='match_id',
    how='left'
)

# per batter per match death chase stats
s2_innings = pressure_s2.groupby(
    ['batter', 'match_id']
).agg(
    death_runs=('runs_off_bat', 'sum'),
    death_balls=('runs_off_bat', 'count'),
    chase_won=('chase_won', 'first')
).reset_index()

# per batter overall death chase stats
s2_stats = s2_innings.groupby('batter').agg(
    s2_innings=('match_id', 'count'),
    s2_runs=('death_runs', 'sum'),
    s2_balls=('death_balls', 'sum'),
    s2_wins=('chase_won', 'sum')
).reset_index()

# compute metrics
s2_stats['s2_sr'] = (
    s2_stats['s2_runs'] / s2_stats['s2_balls'] * 100
).round(2)

s2_stats['s2_win_pct'] = (
    s2_stats['s2_wins'] / s2_stats['s2_innings'] * 100
).round(2)

# combined score
s2_stats['s2_score'] = (
    s2_stats['s2_sr'] * s2_stats['s2_win_pct'] / 100
).round(2)

# updated minimum - 5 innings
s2_stats = s2_stats[s2_stats['s2_innings'] >= threshold]

print(f"\nBatters with {threshold:.0f}+ death chase innings: {len(s2_stats)}")
print("\nTop 10 by combined death chase score:")
print(s2_stats.sort_values('s2_score', ascending=False)[
    ['batter', 's2_innings', 's2_runs', 's2_sr', 's2_win_pct', 's2_score']
].head(10))

Total chase matches: 2477
Chases won: 1180
Chase win rate: 47.6%

Pressure situation 2 balls: 42108
Unique batters: 1016

Batters with 4+ death chase innings: 348

Top 10 by combined death chase score:
              batter  s2_innings  s2_runs   s2_sr  s2_win_pct  s2_score
117          BC Lara           5       45  150.00      100.00    150.00
178         CL White           4       84  144.83      100.00    144.83
467       KJ O'Brien           7      151  160.64       85.71    137.68
153  C de Grandhomme           6      123  161.84       83.33    134.86
365   Iftikhar Ahmed           4       88  166.04       75.00    124.53
330      HM Nicholls           6       81  147.27       83.33    122.72
966          V Kohli          24      516  146.18       83.33    121.81
225      DJ Mitchell           6      117  140.96       83.33    117.46
469         KL Rahul           8      149  134.23       87.50    117.45
281        G Gambhir           4       57  116.33      100.00    116.33


In [83]:
# 1. what is threshold value
print(f"Threshold value: {threshold}")

# 2. check Lara's data
lara = s2_innings[s2_innings['batter'] == 'BC Lara']
print(lara)

Threshold value: 4.0
      batter match_id  death_runs  death_balls  chase_won
423  BC Lara   249758           1            3          1
424  BC Lara   256610           0            1          1
425  BC Lara   267708           6            3          1
426  BC Lara    64865          35           17          1
427  BC Lara    66309           3            6          1


In [84]:
# statistical analysis of death_balls per innings
print("Death balls per innings distribution:")
print(s2_innings['death_balls'].describe())
print(f"\nMedian: {s2_innings['death_balls'].median()}")
print(f"Mean: {s2_innings['death_balls'].mean():.2f}")
print(f"25th percentile: {s2_innings['death_balls'].quantile(0.25)}")
print(f"75th percentile: {s2_innings['death_balls'].quantile(0.75)}")

# statistical analysis of innings per batter
print("\n\nInnings per batter distribution:")
innings_per_batter = s2_innings.groupby('batter')['match_id'].count()
print(innings_per_batter.describe())
print(f"\nMedian: {innings_per_batter.median()}")
print(f"Mean: {innings_per_batter.mean():.2f}")
print(f"25th percentile: {innings_per_batter.quantile(0.25)}")
print(f"75th percentile: {innings_per_batter.quantile(0.75)}")

Death balls per innings distribution:
count    3808.000000
mean       11.057773
std         8.467110
min         1.000000
25%         4.000000
50%         9.000000
75%        16.000000
max        48.000000
Name: death_balls, dtype: float64

Median: 9.0
Mean: 11.06
25th percentile: 4.0
75th percentile: 16.0


Innings per batter distribution:
count    1016.000000
mean        3.748031
std         4.099921
min         1.000000
25%         1.000000
50%         2.000000
75%         5.000000
max        40.000000
Name: match_id, dtype: float64

Median: 2.0
Mean: 3.75
25th percentile: 1.0
75th percentile: 5.0


## Situation 2 — Death Chase (PPI Component 2)

### What this measures
How a batter performs in the final overs of a high-pressure chase.
Specifically: balls faced in overs 40-50 when chasing a target of 250+,
measuring both scoring speed AND match-winning contribution.

### Why this situation matters
Any batter can score in a low-pressure chase of 180.
The truly great finishers score fast AND win matches
when target is 250+ and every ball counts.
This separates genuine death specialists from comfortable scorers.

### Filter conditions
| Condition | Value | Reason |
|---|---|---|
| innings | == 2 | Chasing innings only |
| over_num | >= 40 | Death overs only |
| target | >= 250 | High pressure chase only |
| wides | == 0 | Legal deliveries only |
| death_balls per innings | >= 9 (median) | Genuine death batting only |

### Why target 250+ specifically
Below 250, death over batting is relatively low pressure.
Team can afford to lose wickets and still win.
Above 250, every single ball in overs 40-50 is critical.
250 represents the inflection point in ODI pressure analysis.

### Why NOT strike rate alone
Initial approach used strike rate only.
Problem identified: high SR in a losing chase means nothing.

Example:
- Batter scores 60 off 30 balls (SR 200) but team loses
- vs batter scores 40 off 28 balls (SR 143) and team wins
- Second batter is more valuable despite lower SR

### Metric evolution
| Version | Metric | Problem |
|---|---|---|
| V1 | Strike rate only | Rewarded losing contributions |
| V2 | SR × Win% | Better but small sample issues |
| V3 | SR × Win% with statistical thresholds | Current — most robust |

### Final metric: Death Chase Score = SR × Win% / 100
| Component | What it measures |
|---|---|
| s2_sr | How fast did they score in death overs? |
| s2_win_pct | How often did team win those matches? |
| s2_score | Combined — fast scoring that led to wins |

### Statistically derived thresholds
All thresholds computed from actual data distribution.
No arbitrary values used.

| Threshold | Method | Value | Reason |
|---|---|---|---|
| Min balls per innings | Median (50th percentile) | 9 balls | Removes bottom half low-exposure innings |
| Min qualifying innings | 75th percentile | 5 innings | Only top 25% by innings count qualify |

### Why median for balls, 75th percentile for innings
Balls per innings → median:
- Half of all innings had less than 9 death balls
- These are not genuine death batting situations
- Median threshold removes casual appearances

Qualifying innings → 75th percentile:
- 75% of batters had fewer than 5 qualifying innings
- Only top 25% qualify as genuine death specialists
- Ensures statistical reliability of win percentage

### Filter evolution — fixing BC Lara problem
Original filter (minimum 5 innings count only) had a flaw.

BC Lara example:
- 5 innings but faced only 1, 3, 3, 6, 17 balls per innings
- All 5 matches won → 100% win rate → inflated s2_score
- Not a genuine death batter — just happened to be on field

Fix: minimum 9 balls per qualifying innings (median threshold)
Lara's 1, 3, 3, 6 ball innings all removed
Only his 17-ball innings remains → below 5 innings minimum
Lara correctly excluded from death chase specialists

### Key cricket insights
Kohli #2 with 20 innings and 85% win rate is the most
reliable death chaser in the dataset — large sample, high win rate.
KJ O'Brien #1 despite only 6 innings — known for
Ireland's famous World Cup upsets, each one a high-pressure chase.
Raina #4 with 92.86% win rate — India's most reliable finisher,
rarely lost when he batted in death overs.
Williamson #9 with 100% win rate but only 5 innings —
statistically valid (meets 75th percentile threshold) but
acknowledged as small sample limitation.

### Known limitation
Win percentage reliability increases with innings count.
Batters with exactly 5 qualifying innings and 100% win rate
may have slightly inflated scores.
Future improvement: weight win% by innings count using
Bayesian shrinkage toward population mean.

In [85]:
# statistically derived thresholds
min_balls_per_innings = int(s2_innings['death_balls'].median())
min_qualifying_innings = int(
    s2_innings.groupby('batter')['match_id'].count().quantile(0.75)
)

print(f"Minimum balls per innings (median): {min_balls_per_innings}")
print(f"Minimum qualifying innings (75th percentile): {min_qualifying_innings}")

# apply minimum balls per innings filter
s2_innings_filtered = s2_innings[
    s2_innings['death_balls'] >= min_balls_per_innings
]

print(f"Innings with {min_balls_per_innings}+ death balls: {len(s2_innings_filtered)}")

# per batter stats
s2_stats = s2_innings_filtered.groupby('batter').agg(
    s2_innings=('match_id', 'count'),
    s2_runs=('death_runs', 'sum'),
    s2_balls=('death_balls', 'sum'),
    s2_wins=('chase_won', 'sum')
).reset_index()

# metrics
s2_stats['s2_sr'] = (
    s2_stats['s2_runs'] / s2_stats['s2_balls'] * 100
).round(2)

s2_stats['s2_win_pct'] = (
    s2_stats['s2_wins'] / s2_stats['s2_innings'] * 100
).round(2)

s2_stats['s2_score'] = (
    s2_stats['s2_sr'] * s2_stats['s2_win_pct'] / 100
).round(2)

# apply minimum qualifying innings filter
s2_stats = s2_stats[s2_stats['s2_innings'] >= min_qualifying_innings]

print(f"Batters with {min_qualifying_innings}+ qualifying innings: {len(s2_stats)}")
print("\nTop 10 by death chase score:")
print(s2_stats.sort_values('s2_score', ascending=False)[
    ['batter', 's2_innings', 's2_runs',
     's2_sr', 's2_win_pct', 's2_score']
].head(10))

Minimum balls per innings (median): 9
Minimum qualifying innings (75th percentile): 5
Innings with 9+ death balls: 1965
Batters with 5+ qualifying innings: 106

Top 10 by death chase score:
            batter  s2_innings  s2_runs   s2_sr  s2_win_pct  s2_score
346     KJ O'Brien           6      151  162.37       83.33    135.30
708        V Kohli          20      501  148.22       85.00    125.99
54    Abdul Razzaq           9      283  156.35       77.78    121.61
610       SK Raina          14      370  130.74       92.86    121.41
293        JE Root           8      215  135.22       87.50    118.32
196     EJG Morgan          11      269  143.85       81.82    117.70
291    JDS Neesham           5      124  137.78       80.00    110.22
348       KL Rahul           5      139  136.27       80.00    109.02
355  KS Williamson           5       93  104.49      100.00    104.49
340  KC Sangakkara           9      218  133.74       77.78    104.02


In [86]:
import glob as gl

WC_PATH = r"D:\CricMetric-AI\data\raw\worldcup"
wc_files = gl.glob(os.path.join(WC_PATH, "*.csv"))

wc_match_ids = [
    os.path.basename(f).replace('.csv', '')
    for f in wc_files
]

print(f"World Cup match IDs loaded: {len(wc_match_ids)}")

World Cup match IDs loaded: 265


## Situation 3 — World Cup Performance (PPI Component 3)

### What this measures
How a batter performs specifically in ICC Cricket World Cup matches.
World Cup = highest pressure tournament in ODI cricket.
Every match carries national expectations, legacy implications,
and knockout consequences that bilateral series never replicate.

### Why World Cup deserves its own situation
Pressure in World Cup is categorically different from regular ODIs:
- Elimination pressure — lose and go home
- National expectations — billions watching
- Legacy defining — careers remembered by World Cup moments
- Best players from every nation at their peak simultaneously

A batter averaging 45 in bilateral series but 60 in World Cups
is fundamentally more valuable than one who does the reverse.

### Data source
265 ICC Men's Cricket World Cup matches from Cricsheet.
Match IDs extracted from separate World Cup CSV dataset.
Cross-referenced with main ODI dataset to get ball-by-ball detail.

### Filter conditions
| Condition | Value | Reason |
|---|---|---|
| match_id | in wc_match_ids | World Cup matches only |
| wides | == 0 | Legal deliveries only |

### Why NOT strike rate for World Cup
Unlike death chases, World Cup innings have varying contexts:
- Chasing 180 comfortably → low SR acceptable
- Defending 320 → anchor role more valuable than aggression
- What matters = did they score consistently AND significantly?

Strike rate would unfairly punish anchors and reward sloggers
in matches that were already won comfortably.

### Metrics used
| Metric | Formula | What it captures |
|---|---|---|
| s3_avg | total_runs / dismissals | Scoring volume per innings |
| s3_consistency | % innings with 30+ runs | Reliability across matches |
| s3_50plus | count of 50+ innings | Match-defining contributions |


In [87]:
# Situation 3: World Cup performance
pressure_s3 = master_df[
    (master_df['match_id'].isin(wc_match_ids)) &
    (master_df['wides'] == 0)
].copy()

print(f"World Cup balls: {len(pressure_s3)}")
print(f"Unique batters: {pressure_s3['batter'].nunique()}")

# runs per innings in WC
s3_innings = pressure_s3.groupby(
    ['batter', 'match_id', 'innings']
).agg(
    innings_runs=('runs_off_bat', 'sum'),
    innings_balls=('runs_off_bat', 'count'),
    got_out=('is_wicket', 'max')
).reset_index()

# per batter WC stats
s3_stats = s3_innings.groupby('batter').agg(
    s3_innings=('match_id', 'count'),
    s3_runs=('innings_runs', 'sum'),
    s3_dismissals=('got_out', 'sum'),
    s3_50plus=('innings_runs', lambda x: (x >= 50).sum())
).reset_index()

# average
s3_stats['s3_avg'] = (
    s3_stats['s3_runs'] /
    s3_stats['s3_dismissals'].replace(0, 1)
).round(2)

# consistency
s3_consistency = s3_innings.groupby('batter').apply(
    lambda x: (x['innings_runs'] >= 30).sum() / len(x) * 100
).reset_index()
s3_consistency.columns = ['batter', 's3_consistency']
s3_consistency['s3_consistency'] = s3_consistency['s3_consistency'].round(2)

s3_stats = s3_stats.merge(s3_consistency, on='batter')
# statistical analysis for S3
print("S3 innings per batter distribution:")
s3_innings_count = s3_innings.groupby('batter')['match_id'].count()
print(s3_innings_count.describe())
print(f"Mean:            {s3_innings_count.mean():.2f}")
print(f"Median:          {s3_innings_count.median():.2f}")
print(f"75th percentile: {s3_innings_count.quantile(0.75):.2f}")



World Cup balls: 137817
Unique batters: 695
S3 innings per batter distribution:
count    695.000000
mean       6.728058
std        6.058029
min        1.000000
25%        2.000000
50%        5.000000
75%        8.500000
max       35.000000
Name: match_id, dtype: float64
Mean:            6.73
Median:          5.00
75th percentile: 8.50


In [88]:
min_s3_innings = int(s3_innings_count.quantile(0.75)) + 1  
s3_stats = s3_stats[s3_stats['s3_innings'] >= min_s3_innings]

print(f"\nBatters with {min_s3_innings}+ WC innings: {len(s3_stats)}")
print("\nTop 10 by World Cup average:")
print(s3_stats.sort_values('s3_avg', ascending=False)[
    ['batter', 's3_innings', 's3_runs', 's3_avg', 
     's3_consistency', 's3_50plus']
].head(10))


Batters with 9+ WC innings: 174

Top 10 by World Cup average:
               batter  s3_innings  s3_runs  s3_avg  s3_consistency  s3_50plus
11          A Symonds          13      515   85.83           38.46          4
230          HH Gibbs          14      726   72.60           71.43          7
17     AB de Villiers          22     1207   71.00           59.09         10
429       Mahmudullah          20      894   68.77           50.00          6
512        R Ravindra           9      546   68.25           66.67          5
403         MJ Clarke          21      888   63.43           57.14          8
592           SS Iyer          10      505   63.12           60.00          5
523         RG Sharma          26     1443   62.74           69.23         12
531  RN ten Doeschate           9      435   62.14           55.56          5
342     KS Williamson          24     1055   62.06           66.67          7


In [89]:
kohli_wc = s3_stats[s3_stats['batter'] == 'V Kohli']
dhoni_wc=s3_stats[s3_stats['batter'] == 'MS Dhoni']
print(kohli_wc)
print(dhoni_wc)

      batter  s3_innings  s3_runs  s3_dismissals  s3_50plus  s3_avg  \
664  V Kohli          35     1673             28         15   59.75   

     s3_consistency  
664           65.71  
       batter  s3_innings  s3_runs  s3_dismissals  s3_50plus  s3_avg  \
424  MS Dhoni          24      752             17          5   44.24   

     s3_consistency  
424           45.83  


## Situation 4 — Chase Recovery (PPI Component 4)

### What this measures
How a batter performs when their team loses early wickets
in a chase AND the team still wins the match.

This captures the rarest and most valuable skill in ODI batting:
the ability to rescue a chase that looked lost and win anyway.

### Why this situation is unique
S4 combines two difficult conditions simultaneously:
1. Team is in trouble (3+ wickets down early in chase)
2. Team wins despite the trouble

This means every ball in S4 represents a genuine rescue act.
The batter not only survived the collapse but contributed
enough to turn a losing position into a win.

### How S4 differs from other situations
| Situation | Context | Key difference |
|---|---|---|
| S1 | First innings collapse | Rebuild total, no target |
| S2 | Death overs chase | End game pressure, team intact |
| S3 | World Cup matches | Tournament pressure, any situation |
| S4 | Chase collapse + win | Must rescue AND win, second innings |

S4 is harder than S1 because:
- You have a target and a clock
- Every wicket lost makes target harder
- No margin for error — collapse already happened

S4 is different from S2 because:
- S2 = end game specialist (overs 40-50)
- S4 = rescue specialist (early collapse, overs 0-25)
- Different skills, different players excel in each

### Filter conditions
| Condition | Value | Reason |
|---|---|---|
| innings | == 2 | Chasing innings only |
| match_id | in won_chases | Only won chases included |
| cumulative_wickets | >= 3 | Early collapse happened |
| over_num | < 25 | Early in chase (not death overs) |
| wides | == 0 | Legal deliveries only |
| balls per innings | >= 10 (median) | Genuine recovery contribution |

### Why won_chases filter makes win% redundant
Every match in S4 is already a won chase.
Win% = 100% for all batters → useless as differentiator.
Instead we measure HOW MUCH they contributed during collapse.

### Metric: S4 Score = Average × Strike Rate / 100
| Component | Reason |
|---|---|
| s4_avg | Did they score enough runs to matter? |
| s4_sr | Did they score fast enough to keep chase alive? |
| combined | Both needed simultaneously |


### Statistically derived thresholds
Two separate thresholds applied:

**Threshold 1 — Minimum balls per qualifying innings:**
Distribution of balls per S4 innings:
| Statistic | Value |
|---|---|
| Mean | 14.95 balls |
| Median | 10.0 balls |
| 25th percentile | 1.0 ball |
| 75th percentile | 23.0 balls |

Threshold = median = 10 balls minimum per innings.
Removes bottom 50% of low-exposure recovery appearances.
A batter who faced 1-3 balls in a recovery situation
is not a genuine recovery specialist — just happened to be there.

**Threshold 2 — Minimum qualifying innings:**
Distribution of qualifying innings per batter:
| Statistic | Value |
|---|---|
| Mean | 3.56 innings |
| Median | 2.0 innings |
| 75th percentile | 4.0 innings |

Threshold = 75th percentile = 4 innings minimum.
Only top 25% by qualifying innings count.
Ensures statistical reliability of average and SR calculations.

### Filter evolution
Original: minimum 5 innings (arbitrary)
Problem 1: included innings with 1-3 balls (not genuine recovery)
Problem 2: threshold not statistically justified

Fix: two-stage statistical filter
Stage 1: remove innings with < 10 balls (median threshold)
Stage 2: require 4+ qualifying innings (75th percentile)
Result: 75 genuine chase recovery specialists identified

### Known limitation
S4 only covers chases that were eventually WON.
This means we have no data on batters who made valiant
recovery attempts in losing causes.
A batter who scored 90 in a losing chase from 30/4
gets no credit in S4 — only winners are measured.
This is intentional: we reward match-winning recoveries,
not brave but ultimately unsuccessful ones.
Future improvement: create a separate partial credit score
for recovery attempts in losing causes, weighted lower.

In [90]:
# first get matches where chase was won
# we already have match_outcomes from S2
won_chases = match_outcomes[
    match_outcomes['chase_won'] == 1
]['match_id'].tolist()

print(f"Total won chases: {len(won_chases)}")

# filter: innings 2, early collapse, won matches only
pressure_s4 = innings2[
    (innings2['cumulative_wickets'] >= 3) &
    (innings2['over_num'] < 25) &
    (innings2['match_id'].isin(won_chases)) &
    (innings2['wides'] == 0)
].copy()

print(f"S4 pressure balls: {len(pressure_s4)}")
print(f"Unique batters: {pressure_s4['batter'].nunique()}")

# runs per innings per batter in S4
s4_innings = pressure_s4.groupby(
    ['batter', 'match_id']
).agg(
    s4_runs=('runs_off_bat', 'sum'),
    s4_balls=('runs_off_bat', 'count'),
    got_out=('is_wicket', 'max')
).reset_index()

# per batter overall S4 stats
s4_stats = s4_innings.groupby('batter').agg(
    s4_innings=('match_id', 'count'),
    s4_total_runs=('s4_runs', 'sum'),
    s4_total_balls=('s4_balls', 'sum'),
    s4_dismissals=('got_out', 'sum')
).reset_index()

# average in recovery situations
s4_stats['s4_avg'] = (
    s4_stats['s4_total_runs'] /
    s4_stats['s4_dismissals'].replace(0, 1)
).round(2)

# strike rate too
s4_stats['s4_sr'] = (
    s4_stats['s4_total_runs'] /
    s4_stats['s4_total_balls'] * 100
).round(2)

# combined score avg × SR / 100
s4_stats['s4_score'] = (
    s4_stats['s4_avg'] * s4_stats['s4_sr'] / 100
).round(2)
# statistical analysis for S4
print("S4 innings per batter distribution:")
s4_innings_count = s4_innings.groupby('batter')['match_id'].count()
print(s4_innings_count.describe())
print(f"Mean:            {s4_innings_count.mean():.2f}")
print(f"Median:          {s4_innings_count.median():.2f}")
print(f"75th percentile: {s4_innings_count.quantile(0.75):.2f}")


Total won chases: 1180
S4 pressure balls: 29473
Unique batters: 554
S4 innings per batter distribution:
count    554.000000
mean       3.557762
std        4.155793
min        1.000000
25%        1.000000
50%        2.000000
75%        4.000000
max       30.000000
Name: match_id, dtype: float64
Mean:            3.56
Median:          2.00
75th percentile: 4.00


In [91]:
min_s4_innings = int(s4_innings_count.quantile(0.75)) 
s4_stats = s4_stats[s4_stats['s4_innings'] >= min_s4_innings]

print(f"\nBatters with 5+ recovery innings: {len(s4_stats)}")
print("\nTop 10 by chase recovery score:")
print(s4_stats.sort_values('s4_score', ascending=False)[
    ['batter', 's4_innings', 's4_total_runs',
     's4_avg', 's4_sr', 's4_score']
].head(10))


Batters with 5+ recovery innings: 172

Top 10 by chase recovery score:
                    batter  s4_innings  s4_total_runs  s4_avg   s4_sr  \
384  Nazmul Hossain Shanto           4             98    98.0  122.50   
3               A Flintoff          11            255    85.0  118.60   
265           KIC Asalanka           8            218   109.0   86.51   
12          AB de Villiers          18            376    94.0   96.91   
260             KA Pollard           7            150    75.0  104.17   
320           MG Bracewell           5             85    85.0   90.43   
240        JN Loftie-Eaton           6             87    87.0   87.88   
268               KL Rahul           6            116   116.0   61.38   
69             BB McCullum           5             98    98.0   68.53   
350           Milind Kumar           4             63    63.0  103.28   

     s4_score  
384    120.05  
3      100.81  
265     94.30  
12      91.10  
260     78.13  
320     76.87  
240     76.4

In [92]:
# statistical analysis of balls per S4 innings
print("S4 balls per innings distribution:")
print(s4_innings['s4_balls'].describe())
print(f"Median: {s4_innings['s4_balls'].median()}")
print(f"Mean: {s4_innings['s4_balls'].mean():.2f}")

S4 balls per innings distribution:
count    1971.000000
mean       14.953323
std        15.691533
min         1.000000
25%         1.000000
50%        10.000000
75%        23.000000
max        76.000000
Name: s4_balls, dtype: float64
Median: 10.0
Mean: 14.95


In [93]:
# statistically derived thresholds for S4
min_s4_balls_per_innings = int(s4_innings['s4_balls'].median())  # = 10
min_s4_innings = int(s4_innings_count.quantile(0.75))  # = 4

print(f"S4 min balls per innings (median): {min_s4_balls_per_innings}")
print(f"S4 min qualifying innings (75th percentile): {min_s4_innings}")

# apply minimum balls per innings filter
s4_innings_filtered = s4_innings[
    s4_innings['s4_balls'] >= min_s4_balls_per_innings
]

print(f"Innings with {min_s4_balls_per_innings}+ recovery balls: {len(s4_innings_filtered)}")

# per batter stats
s4_stats = s4_innings_filtered.groupby('batter').agg(
    s4_innings=('match_id', 'count'),
    s4_total_runs=('s4_runs', 'sum'),
    s4_total_balls=('s4_balls', 'sum'),
    s4_dismissals=('got_out', 'sum')
).reset_index()

# metrics
s4_stats['s4_avg'] = (
    s4_stats['s4_total_runs'] /
    s4_stats['s4_dismissals'].replace(0, 1)
).round(2)

s4_stats['s4_sr'] = (
    s4_stats['s4_total_runs'] /
    s4_stats['s4_total_balls'] * 100
).round(2)

s4_stats['s4_score'] = (
    s4_stats['s4_avg'] * s4_stats['s4_sr'] / 100
).round(2)

# apply minimum qualifying innings
s4_stats = s4_stats[s4_stats['s4_innings'] >= min_s4_innings]

print(f"Batters with {min_s4_innings}+ qualifying innings: {len(s4_stats)}")
print("\nTop 10 by chase recovery score:")
print(s4_stats.sort_values('s4_score', ascending=False)[
    ['batter', 's4_innings', 's4_total_runs',
     's4_avg', 's4_sr', 's4_score']
].head(10))

S4 min balls per innings (median): 10
S4 min qualifying innings (75th percentile): 4
Innings with 10+ recovery balls: 997
Batters with 4+ qualifying innings: 75

Top 10 by chase recovery score:
             batter  s4_innings  s4_total_runs  s4_avg   s4_sr  s4_score
2        A Flintoff           7            245   245.0  123.74    303.16
213     LRPL Taylor          12            260   260.0   89.97    233.92
187    KIC Asalanka           6            217   217.0   89.30    193.78
43        BA Stokes           5            174   174.0  107.41    186.89
292       RG Sharma           9            220   220.0   80.29    176.64
7    AB de Villiers          14            364   182.0   96.55    175.72
161         JE Root           8            186   186.0   84.93    157.97
183      KA Pollard           5            144   144.0  109.09    157.09
105      EJG Morgan          17            369   184.5   76.40    140.96
238      MN Samuels           4            131   131.0   94.93    124.36


# PPI — Pressure Performance Index

## Overview
PPI measures how batters perform specifically under pressure situations.
Traditional cricket ratings reward volume — career averages and total runs.
PPI rewards value — runs scored when they matter most.

A batter averaging 45 in comfortable situations is fundamentally different
from one averaging 45 when team is 3 down, chasing 280, in a World Cup knockout.
PPI captures this difference. Traditional ratings do not.

## Why PPI alone is not the final ranking
PPI is one of three components in the final player score:
- PPI (35%) → pressure performance
- CFS (35%) → clutch factor, match-winning contributions
- ENS (30%) → era-adjusted overall batting quality

PPI alone would over-reward pressure specialists who lack overall quality.
Combined with CFS and ENS, it forms a complete picture of batting greatness.

## Architecture — 4 pressure situations

| Situation | Description | Metric | Weight |
|---|---|---|---|
| S1 | Early collapse, innings 1 | Strike Rate | 25% |
| S2 | Death chase, overs 40-50 | SR × Win% | 30% |
| S3 | World Cup performance | Avg + Consistency | 25% |
| S4 | Chase recovery, early collapse won | Avg × SR | 20% |

## Minimum career balls threshold

### Problem with arbitrary thresholds
Original threshold = 500 balls (arbitrary, not justified).
A player with 499 balls excluded, one with 501 included —
no statistical reason for this cutoff.

### Statistical derivation
Distribution of total career balls across all 1,897 batters:
| Statistic | Value |
|---|---|
| Count | 1,897 batters |
| Mean | 694.94 balls |
| Median | 169.00 balls |
| 25th percentile | 41.00 balls |
| 75th percentile | 584.00 balls |
| Max | 15,652 balls |

Threshold = 75th percentile = 584 balls minimum.

### Why 75th percentile
We are ranking Top 100 ODI batters of all time.
This requires genuine established ODI batters, not occasional players.
Removes batters with tiny sample sizes.

## Missing data handling
Batters absent from a situation receive population mean score.
Not zero — zero would unfairly punish batters who never
played in that specific situation.

Example:
- A top order batter rarely faces S4 (chase recovery)
- Not because they are bad — just never needed to rescue
- Population mean gives them neutral score, not penalty

## Statistical threshold summary

| Situation | Threshold Type | Method | Value |
|---|---|---|---|
| S1 | Min career pressure balls | Mean | 60 balls |
| S2 | Min balls per innings | Median | 9 balls |
| S2 | Min qualifying innings | 75th percentile | 5 innings |
| S3 | Min World Cup innings | 75th percentile + 1 | 9 innings |
| S4 | Min balls per innings | Median | 10 balls |
| S4 | Min qualifying innings | 75th percentile | 4 innings |
| PPI | Min career total balls | 75th percentile | 584 balls |

All thresholds derived from actual data distributions.
No arbitrary values used anywhere in the pipeline.

## Weights justification

| Situation | Weight | Reason |
|---|---|---|
| S1 (25%) | Early collapse | Common situation, large sample, first innings |
| S2 (30%) | Death chase | Highest match-winning impact, most visible pressure |
| S3 (25%) | World Cup | Tournament defines legacies, rare opportunity |
| S4 (20%) | Chase recovery | Rarest skill, smaller sample size |

S2 weighted highest because death chase performance
is the most visible and impactful pressure situation in ODI cricket.
Matches are won or lost in overs 40-50 more than any other phase.
S4 weighted lowest because sample size is inherently smaller —
early collapse wins are rarer than other situations.

## Final PPI formula

PPI = (0.25 × s1_norm) +

(0.30 × s2_norm) +
(0.20 × s3_norm) +

(0.05 × s3_con_norm) +

(0.20 × s4_norm)

Note: S3 contributes 25% total split as:
- s3_norm (average) = 20%
- s3_con_norm (consistency) = 5%

## Data scope limitation
Ball-by-ball data available from 2002 onwards only.
Pre-2002 legends (Sachin 1989-2001, Lara 1990-2001) are
underrepresented in pressure situation samples.
This is partially corrected by ENS (Era Normalisation Score)
which adjusts for career span and era scoring conditions.
Pre-2002 performance captured through career averages in ENS component.



In [94]:
from sklearn.preprocessing import MinMaxScaler
# statistically derived career balls threshold

total_balls = master_df[
    master_df['wides'] == 0
].groupby('batter')['runs_off_bat'].count().reset_index()
total_balls.columns = ['batter', 'total_balls']

# statistically derived career balls threshold
min_career_balls = int(total_balls['total_balls'].quantile(0.75))
print(f"Minimum career balls (75th percentile): {min_career_balls}")

qualified = total_balls[total_balls['total_balls'] >= min_career_balls]
print(f"Qualified batters: {len(qualified)}")

# start PPI dataframe with qualified batters
ppi_df = qualified.copy()

# merge all 4 situations
ppi_df = ppi_df.merge(
    s1_stats[['batter', 's1_sr']], 
    on='batter', how='left'
)
ppi_df = ppi_df.merge(
    s2_stats[['batter', 's2_score']], 
    on='batter', how='left'
)
ppi_df = ppi_df.merge(
    s3_stats[['batter', 's3_avg', 's3_consistency']], 
    on='batter', how='left'
)
ppi_df = ppi_df.merge(
    s4_stats[['batter', 's4_score']], 
    on='batter', how='left'
)

print(f"\nBefore filling nulls:")
print(ppi_df.isnull().sum())

# fill missing with population mean
ppi_df['s1_sr'] = ppi_df['s1_sr'].fillna(ppi_df['s1_sr'].mean())
ppi_df['s2_score'] = ppi_df['s2_score'].fillna(ppi_df['s2_score'].mean())
ppi_df['s3_avg'] = ppi_df['s3_avg'].fillna(ppi_df['s3_avg'].mean())
ppi_df['s3_consistency'] = ppi_df['s3_consistency'].fillna(
    ppi_df['s3_consistency'].mean()
)
ppi_df['s4_score'] = ppi_df['s4_score'].fillna(ppi_df['s4_score'].mean())

print(f"\nAfter filling nulls:")
print(ppi_df.isnull().sum())




Minimum career balls (75th percentile): 584
Qualified batters: 475

Before filling nulls:
batter              0
total_balls         0
s1_sr             243
s2_score          383
s3_avg            314
s3_consistency    314
s4_score          400
dtype: int64

After filling nulls:
batter            0
total_balls       0
s1_sr             0
s2_score          0
s3_avg            0
s3_consistency    0
s4_score          0
dtype: int64


In [95]:
# statistical analysis of total career balls
print("Total career balls per batter distribution:")
print(total_balls['total_balls'].describe())
print(f"Mean:            {total_balls['total_balls'].mean():.2f}")
print(f"Median:          {total_balls['total_balls'].median():.2f}")
print(f"25th percentile: {total_balls['total_balls'].quantile(0.25):.2f}")
print(f"75th percentile: {total_balls['total_balls'].quantile(0.75):.2f}")

Total career balls per batter distribution:
count     1897.000000
mean       694.938851
std       1480.347163
min          1.000000
25%         41.000000
50%        169.000000
75%        584.000000
max      15652.000000
Name: total_balls, dtype: float64
Mean:            694.94
Median:          169.00
25th percentile: 41.00
75th percentile: 584.00


In [96]:
# normalise each component to 0-100
scaler = MinMaxScaler(feature_range=(0, 100))

ppi_df['s1_norm'] = scaler.fit_transform(ppi_df[['s1_sr']])
ppi_df['s2_norm'] = scaler.fit_transform(ppi_df[['s2_score']])
ppi_df['s3_norm'] = scaler.fit_transform(ppi_df[['s3_avg']])
ppi_df['s3_con_norm'] = scaler.fit_transform(ppi_df[['s3_consistency']])
ppi_df['s4_norm'] = scaler.fit_transform(ppi_df[['s4_score']])

# weighted PPI
ppi_df['PPI'] = (
    0.25 * ppi_df['s1_norm'] +
    0.30 * ppi_df['s2_norm'] +
    0.20 * ppi_df['s3_norm'] +
    0.05 * ppi_df['s3_con_norm'] +
    0.20 * ppi_df['s4_norm']
).round(2)

# sort by PPI
ppi_df = ppi_df.sort_values('PPI', ascending=False).reset_index(drop=True)
ppi_df['PPI_rank'] = ppi_df.index + 1

print(f"\nTotal qualified batters: {len(ppi_df)}")
print("\nTop 20 by PPI:")
print(ppi_df[['PPI_rank', 'batter', 'PPI', 
              's1_norm', 's2_norm', 
              's3_norm', 's4_norm',
              'total_balls']].head(20))


Total qualified batters: 475

Top 20 by PPI:
    PPI_rank          batter    PPI     s1_norm     s2_norm     s3_norm  \
0          1         V Kohli  60.94   40.909835   93.118995   68.650078   
1          2         JE Root  60.41   43.937162   87.450111   50.342589   
2          3       BA Stokes  59.27   33.775160   71.574279   68.493809   
3          4       RG Sharma  56.55   27.745050   64.552846   72.244260   
4          5     LRPL Taylor  55.02   30.764196   68.455285   41.543455   
5          6  AB de Villiers  54.99   31.729668   51.825573   82.173338   
6          7        KL Rahul  53.14   29.790542   80.576497   69.227071   
7          8   KS Williamson  53.12   24.136802   77.228381   71.426854   
8          9        SK Raina  53.01   30.682376   89.733925   68.553913   
9         10   KC Sangakkara  51.81   31.034201   76.881005   67.327804   
10        11      KJ O'Brien  51.35   34.576992  100.000000   34.319029   
11        12      EJG Morgan  49.60   33.177876   86.9

In [97]:
print(ppi_df.columns.tolist())

['batter', 'total_balls', 's1_sr', 's2_score', 's3_avg', 's3_consistency', 's4_score', 's1_norm', 's2_norm', 's3_norm', 's3_con_norm', 's4_norm', 'PPI', 'PPI_rank']


In [98]:
kohli = ppi_df[ppi_df['batter'] == 'V Kohli']
print(kohli[['batter', 'PPI', 's1_norm', 's2_norm', 
             's3_norm', 's3_con_norm', 's4_norm']].to_string())

    batter    PPI    s1_norm    s2_norm    s3_norm  s3_con_norm    s4_norm
0  V Kohli  60.94  40.909835  93.118995  68.650078    85.426417  23.894871


In [99]:
# show all columns properly
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print(ppi_df[['PPI_rank', 'batter', 'PPI', 
              's1_norm', 's2_norm', 
              's3_norm', 's3_con_norm',
              's4_norm',
              'total_balls']].head(20).to_string())

    PPI_rank          batter    PPI     s1_norm     s2_norm     s3_norm  s3_con_norm    s4_norm  total_balls
0          1         V Kohli  60.94   40.909835   93.118995   68.650078    85.426417  23.894871        15652
1          2         JE Root  60.41   43.937162   87.450111   50.342589    62.181487  50.053321         8415
2          3       BA Stokes  59.27   33.775160   71.574279   68.493809    73.127925  60.002064         3616
3          4       RG Sharma  56.55   27.745050   64.552846   72.244260    90.002600  56.475971        12300
4          5     LRPL Taylor  55.02   30.764196   68.455285   41.543455    65.002600  76.180811         9791
5          6  AB de Villiers  54.99   31.729668   51.825573   82.173338    76.820073  56.159483         9323
6          7        KL Rahul  53.14   29.790542   80.576497   69.227071    72.230889  20.330937         3596
7          8   KS Williamson  53.12   24.136802   77.228381   71.426854    86.674467  26.478379         8749
8          9       

In [100]:
ppi_df.head(1)

,batter,total_balls,s1_sr,s2_score,s3_avg,s3_consistency,s4_score,s1_norm,s2_norm,s3_norm,s3_con_norm,s4_norm,PPI,PPI_rank
0,V Kohli,15652,75.0,125.99,59.75,65.71,81.93,40.909835,93.118995,68.650078,85.426417,23.894871,60.94,1


In [101]:
ppi_df.to_csv(
    r"D:\CricMetric-AI\data\processed\ppi_scores.csv",
    index=False
)
print(f"Saved. Total batters: {len(ppi_df)}")

Saved. Total batters: 475


In [102]:
print(master_df.shape)
print(ppi_df.shape)

(1349408, 30)
(475, 14)


In [103]:
print(ppi_df.columns.tolist())
print(ppi_df.head())

['batter', 'total_balls', 's1_sr', 's2_score', 's3_avg', 's3_consistency', 's4_score', 's1_norm', 's2_norm', 's3_norm', 's3_con_norm', 's4_norm', 'PPI', 'PPI_rank']
        batter  total_balls  s1_sr  s2_score  s3_avg  s3_consistency  \
0      V Kohli        15652  75.00    125.99   59.75           65.71   
1      JE Root         8415  78.70    118.32   44.52           47.83   
2    BA Stokes         3616  66.28     96.84   59.62           56.25   
3    RG Sharma        12300  58.91     87.34   62.74           69.23   
4  LRPL Taylor         9791  62.60     92.62   37.20           50.00   

   s4_score    s1_norm    s2_norm    s3_norm  s3_con_norm    s4_norm    PPI  \
0     81.93  40.909835  93.118995  68.650078    85.426417  23.894871  60.94   
1    157.97  43.937162  87.450111  50.342589    62.181487  50.053321  60.41   
2    186.89  33.775160  71.574279  68.493809    73.127925  60.002064  59.27   
3    176.64  27.745050  64.552846  72.244260    90.002600  56.475971  56.55   
4    23

# CFS — Clutch Factor Score

## Overview
CFS measures match-winning contributions — not just runs scored,
but runs that actually won matches.

A batter who scores 20 fifties in winning causes is fundamentally
more valuable than one who scores 20 fifties in losing causes.
PPI measures pressure performance; CFS measures match-winning impact.
Together with ENS, these three form the complete player evaluation.

## CFS Components
1. Win% when player scored 50+ (vs baseline team win%)
2. Match-winning innings ratio (top scorer in a win)
3. Big score conversion (50+ scores that led to wins)

In [104]:
team_totals = master_df[master_df['wides']==0].groupby(
    ['match_id', 'innings', 'batting_team']
)['total_runs_ball'].sum().reset_index()
team_totals.columns = ['match_id', 'innings', 'batting_team', 'team_total_runs']

print(team_totals.head(10))
print(f"\nTotal team-innings records: {len(team_totals)}")

  match_id  innings batting_team  team_total_runs
0  1000887        1    Australia              259
1  1000887        2     Pakistan              171
2  1000889        1    Australia              209
3  1000889        2     Pakistan              214
4  1000891        1     Pakistan              255
5  1000891        2    Australia              260
6  1000893        1    Australia              353
7  1000893        2     Pakistan              263
8  1000895        1    Australia              363
9  1000895        2     Pakistan              299

Total team-innings records: 5033


In [105]:
team_totals = master_df.groupby(
    ['match_id', 'innings', 'batting_team']
)['cumulative_runs'].max().reset_index()
team_totals.columns = ['match_id', 'innings', 'batting_team', 'team_total_runs']

print(team_totals.head(10))
print(f"\nTotal team-innings records: {len(team_totals)}")

  match_id  innings batting_team  team_total_runs
0  1000887        1    Australia              268
1  1000887        2     Pakistan              176
2  1000889        1    Australia              220
3  1000889        2     Pakistan              221
4  1000891        1     Pakistan              263
5  1000891        2    Australia              265
6  1000893        1    Australia              353
7  1000893        2     Pakistan              267
8  1000895        1    Australia              369
9  1000895        2     Pakistan              312

Total team-innings records: 5033


In [106]:
# check against actual cricket scorecards - Australia vs Pakistan, 2017
# this was the Brisbane match from our very first sample file
print("Match 1000887 (Australia vs Pakistan, Brisbane 2017):")
print(team_totals[team_totals['match_id']=='1000887'])

Match 1000887 (Australia vs Pakistan, Brisbane 2017):
  match_id  innings batting_team  team_total_runs
0  1000887        1    Australia              268
1  1000887        2     Pakistan              176


In [107]:
# Total 1: real match score (includes everything - for win/loss)
match_score = master_df.groupby(
    ['match_id', 'innings', 'batting_team']
)['cumulative_runs'].max().reset_index()
match_score.columns = ['match_id', 'innings', 'batting_team', 'team_match_total']

# Total 2: pure batting total (runs_off_bat only - for contribution share)
# no need to filter wides since runs_off_bat is already pure
batting_total = master_df.groupby(
    ['match_id', 'innings', 'batting_team']
)['runs_off_bat'].sum().reset_index()
batting_total.columns = ['match_id', 'innings', 'batting_team', 'team_batting_total']

print("Match score (real total, includes extras):")
print(match_score.head())
print("\nBatting total (pure runs_off_bat, no extras):")
print(batting_total.head())

Match score (real total, includes extras):
  match_id  innings batting_team  team_match_total
0  1000887        1    Australia               268
1  1000887        2     Pakistan               176
2  1000889        1    Australia               220
3  1000889        2     Pakistan               221
4  1000891        1     Pakistan               263

Batting total (pure runs_off_bat, no extras):
  match_id  innings batting_team  team_batting_total
0  1000887        1    Australia                 257
1  1000887        2     Pakistan                 162
2  1000889        1    Australia                 202
3  1000889        2     Pakistan                 208
4  1000891        1     Pakistan                 248


In [108]:
# filter to only innings 1 and 2 for main match result
match_score_main = match_score[match_score['innings'].isin([1, 2])]

match_pivot = match_score_main.pivot(
    index='match_id', columns='innings', values='team_match_total'
).reset_index()
match_pivot.columns = ['match_id', 'innings1_total', 'innings2_total']

team_names = match_score_main.pivot(
    index='match_id', columns='innings', values='batting_team'
).reset_index()
team_names.columns = ['match_id', 'team_innings1', 'team_innings2']

match_results = match_pivot.merge(team_names, on='match_id')

match_results['winner'] = match_results.apply(
    lambda row: row['team_innings1'] if row['innings1_total'] > row['innings2_total']
    else row['team_innings2'] if row['innings2_total'] > row['innings1_total']
    else 'Tie',
    axis=1
)

print(match_results.head(10))
print(f"\nTotal matches with results: {len(match_results)}")
print(f"\nResult distribution:")
print(match_results['winner'].value_counts().head())

  match_id  innings1_total  innings2_total team_innings1 team_innings2  \
0  1000887           268.0           176.0     Australia      Pakistan   
1  1000889           220.0           221.0     Australia      Pakistan   
2  1000891           263.0           265.0      Pakistan     Australia   
3  1000893           353.0           267.0     Australia      Pakistan   
4  1000895           369.0           312.0     Australia      Pakistan   
5  1001371           324.0           256.0     Australia   New Zealand   
6  1001373           378.0           262.0     Australia   New Zealand   
7  1001375           264.0           147.0     Australia   New Zealand   
8  1004283           153.0           136.0      Scotland     Hong Kong   
9  1004285           266.0           213.0      Scotland     Hong Kong   

      winner  
0  Australia  
1   Pakistan  
2  Australia  
3  Australia  
4  Australia  
5  Australia  
6  Australia  
7  Australia  
8   Scotland  
9   Scotland  

Total matches with 

In [109]:
# get each player's runs per innings + their team's batting total
player_innings = master_df.groupby(
    ['batter', 'match_id', 'innings', 'batting_team']
).agg(
    player_runs=('runs_off_bat', 'sum'),
    player_balls=('runs_off_bat', 'count'),
    got_out=('is_wicket', 'max')
).reset_index()

# merge team batting total
player_innings = player_innings.merge(
    batting_total, on=['match_id', 'innings', 'batting_team']
)

# compute contribution share
player_innings['contribution_share'] = (
    player_innings['player_runs'] / player_innings['team_batting_total']
).round(4)

# merge match winner info
player_innings = player_innings.merge(
    match_results[['match_id', 'winner']], on='match_id', how='left'
)

# did this player's team win?
player_innings['team_won'] = (
    player_innings['batting_team'] == player_innings['winner']
).astype(int)

print(player_innings.head(10))
print(f"\nTotal player-innings records: {len(player_innings)}")

       batter match_id  innings batting_team  player_runs  player_balls  \
0     A Ashok  1388212        1  New Zealand           10            12   
1  A Athanaze  1373576        2  West Indies           66            65   
2  A Athanaze  1373577        1  West Indies            4            14   
3  A Athanaze  1373578        2  West Indies           45            51   
4  A Athanaze  1375847        1  West Indies            5            11   
5  A Athanaze  1375848        2  West Indies           11            14   
6  A Athanaze  1375849        1  West Indies           32            64   
7  A Athanaze  1377007        2  West Indies           65            48   
8  A Athanaze  1381214        1  West Indies           22            19   
9  A Athanaze  1381215        2  West Indies            6             9   

   got_out  team_batting_total  contribution_share       winner  team_won  
0        1                  94              0.1064   Bangladesh         0  
1        1            

### Component 4: Dominance Score
Measures how often a batter's individual contribution
crossed a scaling threshold based on team total size.

| Team Total Range | Threshold | Reasoning |
|---|---|---|
| < 100 | 25% | Even moderate share matters in small totals |
| 100-200 | 30% | Standard collapse/defensive total |
| 200-300 | 35% | Competitive total, harder to dominate |
| 300+ | 40% | Big total, true dominance needs bigger share |

dominance_count = number of innings where contribution_share
                  crossed the applicable threshold

dominance_rate = dominance_count / total_innings

In [110]:
# filter to only 50+ scores (the "big innings" we care about for clutch)
big_scores = player_innings[player_innings['player_runs'] >= 50].copy()

print(f"Total 50+ innings: {len(big_scores)}")
print(f"Unique batters with 50+: {big_scores['batter'].nunique()}")

# simple win% when scored 50+
simple_cfs = big_scores.groupby('batter').agg(
    fifties_plus=('match_id', 'count'),
    wins_with_50plus=('team_won', 'sum')
).reset_index()

simple_cfs['win_pct_50plus'] = (
    simple_cfs['wins_with_50plus'] / simple_cfs['fifties_plus'] * 100
).round(2)

print("\nTop 10 by win% when scoring 50+:")
print(simple_cfs.sort_values('win_pct_50plus', ascending=False).head(10))

Total 50+ innings: 7040
Unique batters with 50+: 792

Top 10 by win% when scoring 50+:
                batter  fifties_plus  wins_with_50plus  win_pct_50plus
775          Wasim Ali             1                 1           100.0
10            A Sharma             5                 5           100.0
791       Zubayr Hamza             1                 1           100.0
0           A Athanaze             2                 2           100.0
774  Washington Sundar             1                 1           100.0
768        WS Jayantha             1                 1           100.0
779        YBK Jaiswal             1                 1           100.0
314        JD Campbell             1                 1           100.0
312      JAD Lawrenson             1                 1           100.0
343         JP Greaves             1                 1           100.0


In [111]:
print("Fifties_plus distribution:")
print(simple_cfs['fifties_plus'].describe())
print(f"\nMean: {simple_cfs['fifties_plus'].mean():.2f}")
print(f"Median: {simple_cfs['fifties_plus'].median():.2f}")
print(f"75th percentile: {simple_cfs['fifties_plus'].quantile(0.75):.2f}")

Fifties_plus distribution:
count    792.000000
mean       8.888889
std       14.075569
min        1.000000
25%        1.000000
50%        3.000000
75%        9.250000
max      129.000000
Name: fifties_plus, dtype: float64

Mean: 8.89
Median: 3.00
75th percentile: 9.25


In [112]:
min_fifties = int(simple_cfs['fifties_plus'].quantile(0.75)) + 1
print(f"Minimum 50+ scores threshold (75th percentile): {min_fifties}")

simple_cfs_filtered = simple_cfs[simple_cfs['fifties_plus'] >= min_fifties]

print(f"Qualified batters: {len(simple_cfs_filtered)}")
print("\nTop 10 by win% when scoring 50+:")
print(simple_cfs_filtered.sort_values('win_pct_50plus', ascending=False).head(10))

Minimum 50+ scores threshold (75th percentile): 10
Qualified batters: 198

Top 10 by win% when scoring 50+:
           batter  fifties_plus  wins_with_50plus  win_pct_50plus
11      A Symonds            29                28           96.55
81     Aqib Ilyas            11                10           90.91
196     DR Martyn            21                19           90.48
605    RR Rossouw            10                 9           90.00
747    UT Khawaja            14                12           85.71
518      NJ Astle            13                11           84.62
706  Shubman Gill            25                21           84.00
267   HM Nicholls            18                15           83.33
341     JP Duminy            30                25           83.33
783  Yasir Hameed            12                10           83.33


In [113]:
# Component 4: Dominance Score
# scaling threshold based on team total

def get_dominance_threshold(team_total):
    if team_total < 100:
        return 0.25
    elif team_total < 200:
        return 0.30
    elif team_total < 300:
        return 0.35
    else:
        return 0.40

player_innings['dominance_threshold'] = player_innings['team_batting_total'].apply(
    get_dominance_threshold
)

player_innings['is_dominant'] = (
    player_innings['contribution_share'] >= player_innings['dominance_threshold']
).astype(int)

print(f"Total dominant innings: {player_innings['is_dominant'].sum()}")
print(f"Percentage of all innings: {player_innings['is_dominant'].mean()*100:.2f}%")

# per batter dominance stats
dominance_stats = player_innings.groupby('batter').agg(
    total_innings=('match_id', 'count'),
    dominant_innings=('is_dominant', 'sum')
).reset_index()

dominance_stats['dominance_rate'] = (
    dominance_stats['dominant_innings'] / dominance_stats['total_innings'] * 100
).round(2)

print("\nTop 10 by dominance rate:")
print(dominance_stats.sort_values('dominance_rate', ascending=False).head(10))

Total dominant innings: 3309
Percentage of all innings: 7.45%

Top 10 by dominance rate:
            batter  total_innings  dominant_innings  dominance_rate
511       FY Fazal              1                 1           100.0
376     D Daesrath              1                 1           100.0
311     CB Lambert              1                 1           100.0
263      BT Foakes              1                 1           100.0
98        AM Tribe              4                 2            50.0
695    J Figy John              2                 1            50.0
746       JF Smith              2                 1            50.0
945    L Steenkamp              4                 2            50.0
1541    SJ Massiah              2                 1            50.0
280   C Carmichael              2                 1            50.0


In [114]:
print("Total innings distribution (for dominance):")
print(dominance_stats['total_innings'].describe())
print(f"\n75th percentile: {dominance_stats['total_innings'].quantile(0.75):.2f}")

Total innings distribution (for dominance):
count    1897.000000
mean       23.410121
std        37.110501
min         1.000000
25%         3.000000
50%         9.000000
75%        27.000000
max       296.000000
Name: total_innings, dtype: float64

75th percentile: 27.00


In [115]:
min_innings_dominance = int(dominance_stats['total_innings'].quantile(0.90))
print(f"Minimum innings threshold (90th percentile): {min_innings_dominance}")

dominance_filtered = dominance_stats[
    dominance_stats['total_innings'] >= min_innings_dominance
]

print(f"Qualified batters: {len(dominance_filtered)}")
print("\nTop 10 by dominance rate:")
print(dominance_filtered.sort_values('dominance_rate', ascending=False).head(10))

Minimum innings threshold (90th percentile): 63
Qualified batters: 192

Top 10 by dominance rate:
             batter  total_innings  dominant_innings  dominance_rate
1584   SR Tendulkar            144                34           23.61
601       HG Munsey             67                15           22.39
1800        V Kohli            296                66           22.30
665       IJL Trott             65                14           21.54
876      KJ Coetzer             63                13           20.63
1600  ST Jayasuriya            133                27           20.30
99          AN Cook             91                18           19.78
604        HH Gibbs            102                20           19.61
613         HM Amla            174                34           19.54
1764    Tamim Iqbal            215                42           19.53


In [116]:
# find top scorer for each team in each match
top_scorers = player_innings.loc[
    player_innings.groupby(['match_id', 'innings', 'batting_team'])['player_runs'].idxmax()
]

print(f"Total top-scorer records: {len(top_scorers)}")
print(top_scorers[['batter', 'match_id', 'player_runs', 'team_won']].head(10))

# was this top scorer's team also the winner?
top_scorers['was_match_winning_topscore'] = (
    (top_scorers['team_won'] == 1)
).astype(int)

# per batter: how many times were they top scorer in a win
match_winning_stats = top_scorers.groupby('batter').agg(
    times_top_scorer=('match_id', 'count'),
    times_topscore_in_win=('was_match_winning_topscore', 'sum')
).reset_index()

match_winning_stats['match_winning_ratio'] = (
    match_winning_stats['times_topscore_in_win'] / 
    match_winning_stats['times_top_scorer'] * 100
).round(2)

print("\nTop scorer distribution:")
print(match_winning_stats['times_top_scorer'].describe())

Total top-scorer records: 5033
                batter match_id  player_runs  team_won
26139          MS Wade  1000887          100         1
5991        Babar Azam  1000887           33         0
36530        SPD Smith  1000889           60         0
27156  Mohammad Hafeez  1000889           72         1
5993        Babar Azam  1000891           84         0
36531        SPD Smith  1000891          108         1
8723         DA Warner  1000893          130         1
38640    Sharjeel Khan  1000893           74         0
8724         DA Warner  1000895          179         1
5995        Babar Azam  1000895          100         0

Top scorer distribution:
count    775.000000
mean       6.494194
std        9.454209
min        1.000000
25%        1.000000
50%        3.000000
75%        7.000000
max       79.000000
Name: times_top_scorer, dtype: float64


In [117]:
print(f"90th percentile: {match_winning_stats['times_top_scorer'].quantile(0.90):.2f}")

min_top_scorer = int(match_winning_stats['times_top_scorer'].quantile(0.90))
print(f"Minimum threshold: {min_top_scorer}")

match_winning_filtered = match_winning_stats[
    match_winning_stats['times_top_scorer'] >= min_top_scorer
]

print(f"Qualified batters: {len(match_winning_filtered)}")
print("\nTop 10 by match-winning ratio:")
print(match_winning_filtered.sort_values('match_winning_ratio', ascending=False).head(10))

90th percentile: 18.00
Minimum threshold: 18
Qualified batters: 79

Top 10 by match-winning ratio:
             batter  times_top_scorer  times_topscore_in_win  \
9         A Symonds                22                     22   
590      RT Ponting                33                     28   
19     AC Gilchrist                27                     21   
436       MJ Clarke                36                     26   
17   AB de Villiers                50                     36   
438      MJ Guptill                39                     28   
715      TM Dilshan                48                     34   
766     Younis Khan                19                     13   
265         HM Amla                38                     26   
734         V Kohli                79                     54   

     match_winning_ratio  
9                 100.00  
590                84.85  
19                 77.78  
436                72.22  
17                 72.00  
438                71.79  
715    

# CFS — Clutch Factor Score (Final: 3 Components)

## Overview
CFS measures match-winning contributions — not just runs scored,
but runs that translate into team success. This complements PPI
(pressure performance) by adding the dimension of result impact.

## Why 3 components, not the originally proposed acceleration metric
An "innings acceleration" idea (SR change within an innings) was
considered but rejected for CFS — it measures batting technique/style,
not match-winning impact. It overlaps with PPI's pressure situations
and is better suited as a future standalone dashboard visualisation,
not a ranking component.

## Components

### Component 1: Win% when scored 50+
Measures general correlation between big scores and match wins.
Threshold: 10+ fifties (75th percentile of fifties_plus distribution).
Qualified batters: 198

### Component 2: Match-winning innings ratio
Was this player the top scorer for their team specifically in wins?
Threshold: 18+ times as top scorer (90th percentile).
Qualified batters: 79

### Component 3: Dominance Score (renamed from Component 4)
Measures individual contribution dominance independent of match result.
Solves the "weak team era" and "milestone player" problem directly —
captures genuine batting greatness even when team result was poor.

Scaling thresholds based on team total:
| Team Total | Threshold |
|---|---|
| < 100 | 30% |
| 100-200 | 35% |
| 200-300 | 40% |
| 300+ | 45% |

Threshold refined from 75th to 90th percentile (63+ innings) to ensure
comparison only between batters with substantial career sample sizes,
avoiding noise from batters with few innings.
Qualified batters: 192

## Why this addresses the original concern
A legendary batter in a weak team era (e.g. Sachin's early India teams)
shows high Dominance Score even in losses, because it measures
contribution share, not match result. This was specifically validated:
Tendulkar ranks #1 in Dominance Score (23.61%) — proving the metric
correctly identifies individual greatness independent of team weakness.

In [118]:
from sklearn.preprocessing import MinMaxScaler

# get all batters who appear in player_innings
all_cfs_batters = pd.DataFrame(
    player_innings['batter'].unique(), columns=['batter']
)

cfs_df = all_cfs_batters.copy()

# merge all 3 components
cfs_df = cfs_df.merge(
    simple_cfs_filtered[['batter', 'win_pct_50plus']],
    on='batter', how='left'
)
cfs_df = cfs_df.merge(
    match_winning_filtered[['batter', 'match_winning_ratio']],
    on='batter', how='left'
)
cfs_df = cfs_df.merge(
    dominance_filtered[['batter', 'dominance_rate']],
    on='batter', how='left'
)

# fill missing with population mean
cfs_df['win_pct_50plus'] = cfs_df['win_pct_50plus'].fillna(
    cfs_df['win_pct_50plus'].mean()
)
cfs_df['match_winning_ratio'] = cfs_df['match_winning_ratio'].fillna(
    cfs_df['match_winning_ratio'].mean()
)
cfs_df['dominance_rate'] = cfs_df['dominance_rate'].fillna(
    cfs_df['dominance_rate'].mean()
)

# normalise each to 0-100
scaler = MinMaxScaler(feature_range=(0, 100))
cfs_df['c1_norm'] = scaler.fit_transform(cfs_df[['win_pct_50plus']])
cfs_df['c2_norm'] = scaler.fit_transform(cfs_df[['match_winning_ratio']])
cfs_df['c3_norm'] = scaler.fit_transform(cfs_df[['dominance_rate']])

# weighted CFS
cfs_df['CFS'] = (
    0.35 * cfs_df['c1_norm'] +
    0.35 * cfs_df['c2_norm'] +
    0.30 * cfs_df['c3_norm']
).round(2)

# apply same career balls qualification as PPI (584 balls, 75th percentile)
cfs_df = cfs_df.merge(total_balls, on='batter')
cfs_df = cfs_df[cfs_df['total_balls'] >= min_career_balls]

cfs_df = cfs_df.sort_values('CFS', ascending=False).reset_index(drop=True)
cfs_df['CFS_rank'] = cfs_df.index + 1

print(f"Total qualified batters: {len(cfs_df)}")
print("\nTop 20 by CFS:")
print(cfs_df[['CFS_rank', 'batter', 'CFS', 'c1_norm', 'c2_norm', 'c3_norm']].head(20))

Total qualified batters: 475

Top 20 by CFS:
    CFS_rank          batter    CFS     c1_norm     c2_norm     c3_norm
0          1       A Symonds  80.89  100.000000  100.000000   36.298179
1          2    AC Gilchrist  77.93   79.359345   75.630621   78.949598
2          3         V Kohli  75.88   70.544316   65.288440   94.451504
3          4         HM Amla  75.60   79.696532   65.365212   82.761542
4          5       Q de Kock  72.95   75.168593   62.480807   82.592122
5          6    SR Tendulkar  72.58   65.522640   56.130730  100.000000
6          7      RT Ponting  71.90   81.707611   83.384514   47.056332
7          8      MJ Guptill  69.95   75.686416   69.061198   64.294790
8          9  AB de Villiers  69.44   72.880539   69.291511   65.607793
9         10       RG Sharma  68.02   67.509634   62.294363   75.307073
10        11       G Gambhir  67.72   67.750482   63.445931   72.681067
11        12     Tamim Iqbal  67.12   57.418112   63.445931   82.719187
12        13   ST J

In [119]:
cfs_df.to_csv(
    r"D:\CricMetric-AI\data\processed\cfs_scores.csv",
    index=False
)
print(f"CFS saved. Total batters: {len(cfs_df)}")

CFS saved. Total batters: 475
